# HarmonicMoE + learned phase harmonizer + DDP

Финальный сложный pipeline использует два фиксированных нейросетевых этапа:

1. HarmonicMoE строит первичную фазу из spatial, cross-plane Transformer и Fourier experts;
2. первичная фаза проходит через Френель, после чего harmonizer по целям, восстановлениям и ошибкам
   выдаёт одну круговую фазовую поправку.

Это не итеративная оптимизация: число этапов фиксировано, обновления параметров на инференсе отсутствуют.
Обучение запускается через `torchrun`/DDP: один процесс на GPU. Float16 render cache строится один раз до
запуска процессов.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_msssim import ssim
except ImportError:
    print('Installing pytorch-msssim...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v2')
OUTPUT_DIR = Path('/kaggle/working/moe_harmonizer_ddp_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this to last.pt from an attached Kaggle Dataset to resume across sessions.
RESUME_CHECKPOINT = None
BASE_CHECKPOINT = None  # optional best/last checkpoint from SharedDepthUNet

IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)

TRAIN_SEED = 42
MAX_EPOCHS = 30
MIN_EPOCHS = 12
EARLY_STOP_PATIENCE = 6
BATCH_SIZE = 32  # 16 samples per GPU for 2 x T4; try 64 if memory allows
LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
SSIM_WEIGHT = 1.5
NUM_WORKERS = min(2, os.cpu_count() or 1)
LOG_EVERY = 50
CACHE_WORKERS = min(4, os.cpu_count() or 1)
CACHE_DIR = Path('/kaggle/working/render_cache_v2')
GATE_BALANCE_WEIGHT = 0.01

def find_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_root(KAGGLE_DATASET_DIR)
metadata = json.loads((DATA_ROOT / 'dataset_metadata.json').read_text())
if metadata.get('schema_version') != 2:
    raise RuntimeError(f'Expected dataset schema 2, got {metadata.get("schema_version")}')
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'

CONFIG = {
    'architecture': 'HarmonicMoEUNet+PhaseHarmonizer+DDP',
    'dataset_schema': 2,
    'train_seed': TRAIN_SEED,
    'max_epochs': MAX_EPOCHS,
    'min_epochs': MIN_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'min_lr': MIN_LR,
    'weight_decay': WEIGHT_DECAY,
    'ssim_weight': SSIM_WEIGHT,
        'gate_balance_weight': GATE_BALANCE_WEIGHT,
    'image_size': IMAGE_SIZE,
    'wavelength': WAVELENGTH,
    'pixel_pitch': PIXEL_PITCH,
    'zs': ZS,
}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'disabled')
PER_GPU_BATCH = 16
DDP_BASE_CHECKPOINT = None
DDP_RESUME_CHECKPOINT = None


In [ ]:
RESAMPLE = Image.Resampling.LANCZOS

def normalize_image(array: np.ndarray) -> torch.Tensor:
    values = torch.from_numpy(np.asarray(array, dtype=np.float32) / 255.0)
    maximum = values.amax()
    if maximum <= 1e-8:
        raise ValueError('All-black image cannot be normalized')
    return values / maximum

def font_for_size(size: int):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()

def draw_primitive(mask: Image.Image, rng: np.random.Generator, remaining_fraction: float) -> None:
    draw = ImageDraw.Draw(mask)
    target_area = max(12.0, remaining_fraction * IMAGE_SIZE * IMAGE_SIZE * float(rng.uniform(0.25, 0.65)))
    kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'glyph'], p=[0.28, 0.28, 0.16, 0.14, 0.14]))
    aspect = float(np.exp(rng.uniform(np.log(0.45), np.log(2.2))))
    width = int(np.clip(np.sqrt(target_area * aspect), 3, IMAGE_SIZE - 4))
    height = int(np.clip(np.sqrt(target_area / aspect), 3, IMAGE_SIZE - 4))
    x0 = int(rng.integers(2, max(3, IMAGE_SIZE - width - 1)))
    y0 = int(rng.integers(2, max(3, IMAGE_SIZE - height - 1)))
    x1, y1 = min(IMAGE_SIZE - 2, x0 + width), min(IMAGE_SIZE - 2, y0 + height)

    if kind == 'ellipse':
        draw.ellipse((x0, y0, x1, y1), fill=255)
    elif kind == 'rectangle':
        draw.rectangle((x0, y0, x1, y1), fill=255)
    elif kind == 'line':
        line_width = max(1, int(np.clip(target_area / max(width, height), 1, 10)))
        draw.line((x0, y0, x1, y1), fill=255, width=line_width)
    elif kind == 'ring':
        ring_width = max(1, int(np.clip(min(width, height) * 0.12, 1, 9)))
        draw.ellipse((x0, y0, x1, y1), outline=255, width=ring_width)
    else:
        char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
        font = font_for_size(max(10, min(80, height)))
        glyph = Image.new('L', (max(16, width * 2), max(16, height * 2)), 0)
        gd = ImageDraw.Draw(glyph)
        box = gd.textbbox((0, 0), char, font=font)
        gd.text((-box[0], -box[1]), char, fill=255, font=font)
        box = glyph.getbbox()
        if box is not None:
            glyph = glyph.crop(box).resize((max(3, width), max(3, height)), RESAMPLE)
            mask.paste(glyph, (x0, y0), glyph)

def make_mask(seed: int, target_coverage: float) -> tuple[Image.Image, float]:
    best_mask, best_coverage, best_error = None, 0.0, float('inf')
    for attempt in range(8):
        rng = np.random.default_rng(seed + 1009 * attempt)
        mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
        for _ in range(18):
            coverage = float((np.asarray(mask) > 16).mean())
            error = abs(coverage - target_coverage)
            if error < best_error:
                best_mask, best_coverage, best_error = mask.copy(), coverage, error
            if coverage >= target_coverage * 0.96:
                break
            draw_primitive(mask, rng, max(target_coverage - coverage, 0.002))
        coverage = float((np.asarray(mask) > 16).mean())
        error = abs(coverage - target_coverage)
        if error < best_error:
            best_mask, best_coverage, best_error = mask.copy(), coverage, error
        if target_coverage * 0.75 <= coverage <= target_coverage * 1.25:
            return mask, coverage
    return best_mask, best_coverage

def render_synthetic(item: dict, return_coverage: bool = False):
    seed = int(item['seed'])
    rng = np.random.default_rng(seed)
    target = float(item['target_coverage'])
    mask, coverage = make_mask(seed, target)
    mode = item['background_mode']
    if mode == 'dark':
        background = int(rng.integers(0, 19)); foreground = int(rng.integers(max(80, background + 65), 256))
    elif mode == 'light':
        background = int(rng.integers(175, 246)); foreground = int(rng.integers(0, max(1, background - 65)))
    else:
        background = int(rng.integers(25, 101))
        if rng.random() < 0.7:
            foreground = int(rng.integers(min(255, background + 65), 256))
        else:
            foreground = int(rng.integers(0, max(1, background - 20)))
    mask_values = np.asarray(mask, dtype=np.float32) / 255.0
    image = background + mask_values * (foreground - background)
    image = np.clip(np.rint(image), 0, 255).astype(np.uint8)
    tensor = normalize_image(image)
    return (tensor, coverage) if return_coverage else tensor

def load_natural(item: dict, root: Path = BSDS_ROOT) -> torch.Tensor:
    path = root / item['path']
    with Image.open(path) as source:
        image = source.convert('L')
        if item['transform'] == 'full':
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), method=RESAMPLE, color=fill, centering=(0.5, 0.5))
        else:
            crop = item['crop']; size = int(crop['size']); x, y = int(crop['x']), int(crop['y'])
            image = image.crop((x, y, x + size, y + size)).resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        if item.get('hflip', False):
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        angle = int(item.get('rotation', 0))
        if angle:
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=fill)
        return normalize_image(np.asarray(image, dtype=np.uint8))

def load_item(item: dict) -> torch.Tensor:
    return render_synthetic(item) if item['source'] == 'synthetic' else load_natural(item)

class QuadDataset(Dataset):
    def __init__(self, manifest: Path):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        quad = torch.stack([load_item(item) for item in self.records[index]['items']]).to(torch.float32)
        if quad.shape != (4, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f'Invalid quad shape: {tuple(quad.shape)}')
        return quad

class QuadSourceDataset(QuadDataset):
    def __getitem__(self, index):
        record = self.records[index]
        quad = torch.stack([load_item(item) for item in record['items']]).to(torch.float32)
        sources = torch.tensor([1 if item['source'] == 'bsds' else 0 for item in record['items']], dtype=torch.long)
        return quad, sources

## Одноразовый render cache

Эта ячейка временно нагружает CPU, но после неё train/validation читаются из готовых float16 memmap.
Маркер `.complete` защищает от использования недостроенного кэша после прерывания.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU acceleration')
DEVICE_IDS = list(range(torch.cuda.device_count()))
if len(DEVICE_IDS) < 2:
    print('WARNING: only one GPU is visible; DataParallel will not be enabled')
primary_device = torch.device(f'cuda:{DEVICE_IDS[0]}')
print('GPUs:', [torch.cuda.get_device_name(i) for i in DEVICE_IDS])

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(TRAIN_SEED)

def build_cache(split):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    image_path, source_path = CACHE_DIR / f'{split}_images.npy', CACHE_DIR / f'{split}_sources.npy'
    done_path = CACHE_DIR / f'{split}.complete'
    manifest = DATA_ROOT / 'manifests' / f'{split}.jsonl'
    base = QuadSourceDataset(manifest)
    if image_path.exists() and source_path.exists() and done_path.exists():
        images = np.load(image_path, mmap_mode='r')
        if images.shape[0] == len(base):
            print(split, 'cache already exists:', image_path)
            return image_path, source_path
    loader = DataLoader(base, batch_size=64, shuffle=False, num_workers=CACHE_WORKERS,
                        pin_memory=False, persistent_workers=CACHE_WORKERS > 0)
    images = np.lib.format.open_memmap(image_path, mode='w+', dtype=np.float16,
                                       shape=(len(base), 4, IMAGE_SIZE, IMAGE_SIZE))
    sources = np.lib.format.open_memmap(source_path, mode='w+', dtype=np.uint8, shape=(len(base), 4))
    offset, started = 0, time.perf_counter()
    for batch_index, (quad, source) in enumerate(loader, start=1):
        size = quad.shape[0]
        images[offset:offset+size] = quad.numpy().astype(np.float16)
        sources[offset:offset+size] = source.numpy().astype(np.uint8)
        offset += size
        if batch_index == 1 or batch_index % 50 == 0 or batch_index == len(loader):
            print(f'cache {split}: {offset}/{len(base)}, {time.perf_counter()-started:.0f}s', flush=True)
    images.flush(); sources.flush(); done_path.write_text('complete', encoding='utf-8')
    return image_path, source_path

class CachedQuadDataset(Dataset):
    def __init__(self, image_path, source_path):
        self.images = np.load(image_path, mmap_mode='r')
        self.sources = np.load(source_path, mmap_mode='r')
    def __len__(self): return self.images.shape[0]
    def __getitem__(self, index):
        image = torch.from_numpy(np.array(self.images[index], copy=True))
        source = torch.from_numpy(np.array(self.sources[index], copy=True)).long()
        return image, source

train_cache = build_cache('train'); val_cache = build_cache('val')
train_dataset = CachedQuadDataset(*train_cache); val_dataset = CachedQuadDataset(*val_cache)



## Генерация DDP training script

Notebook пишет self-contained Python script, затем следующая ячейка запускает два процесса через
`torch.distributed.run`. Логи обоих процессов синхронизируются NCCL; checkpoints пишет только rank 0.

In [ ]:
DDP_SCRIPT_TEMPLATE = "import csv, json, math, os, random, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom pytorch_msssim import ssim\n\nIMAGE_SIZE=128; WAVELENGTH=532e-9; PIXEL_PITCH=10e-6\nZS=(24.1e-3,25.3e-3,26.5e-3,27.7e-3)\nOUTPUT_DIR=Path('/kaggle/working/moe_harmonizer_ddp_v2'); CACHE_DIR=Path('/kaggle/working/render_cache_v2')\nMAX_EPOCHS=30; MIN_EPOCHS=12; PATIENCE=6; PER_GPU_BATCH=__PER_GPU_BATCH__\nLR=3e-4; MIN_LR=1e-5; WEIGHT_DECAY=1e-4; SSIM_WEIGHT=1.5; GATE_BALANCE_WEIGHT=0.01\nBASE_CHECKPOINT=__BASE_CHECKPOINT__; RESUME_CHECKPOINT=__RESUME_CHECKPOINT__; SEED=42\n\nclass CachedQuadDataset(Dataset):\n    def __init__(self, split):\n        self.images=np.load(CACHE_DIR/f'{split}_images.npy',mmap_mode='r')\n        self.sources=np.load(CACHE_DIR/f'{split}_sources.npy',mmap_mode='r')\n    def __len__(self): return self.images.shape[0]\n    def __getitem__(self,index):\n        return torch.from_numpy(np.array(self.images[index],copy=True)), torch.from_numpy(np.array(self.sources[index],copy=True)).long()\n\nclass DoubleConv(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),\n                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())\n\nclass UpBlock(nn.Module):\n    def __init__(self, cin, skip, cout):\n        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)\n    def forward(self, x, skip):\n        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)\n        return self.block(torch.cat((self.reduce(x), skip), dim=1))\n\nclass BaselineUNet(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)\n        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)\n        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)\n        self.head = nn.Conv2d(32, 1, 1)\n    def forward(self, x):\n        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))\n        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)\n        return self.head(x)\n\nclass DepthEmbedding(nn.Module):\n    def __init__(self, channels):\n        super().__init__(); hidden = max(16, channels // 2)\n        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))\n        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)\n    def forward(self, depths):\n        return self.net(depths[:, None])[:, :, None, None]\n\nclass PlaneFusion(nn.Sequential):\n    def __init__(self, channels):\n        super().__init__(nn.Conv2d(4 * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())\n\nclass SpatialExpert(nn.Module):\n    def __init__(self, channels=512):\n        super().__init__()\n        hidden = channels // 2\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, hidden, 1, bias=False), nn.GroupNorm(8, hidden), nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),\n            nn.GroupNorm(8, hidden), nn.SiLU(), nn.Conv2d(hidden, channels, 1, bias=False),\n        )\n    def forward(self, x):\n        return x + self.net(x)\n\nclass CrossPlaneTransformerExpert(nn.Module):\n    def __init__(self, channels=512, planes=4, height=8, width=8, dim=256):\n        super().__init__()\n        self.planes, self.height, self.width = planes, height, width\n        self.input_projection = nn.Linear(channels, dim)\n        self.position = nn.Parameter(torch.zeros(1, planes * height * width, dim))\n        layer = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=8, dim_feedforward=3 * dim, dropout=0.1,\n            activation='gelu', batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(layer, num_layers=2)\n        self.output_projection = nn.Linear(dim, channels)\n        self.fusion = PlaneFusion(channels)\n        nn.init.trunc_normal_(self.position, std=0.02)\n\n    def forward(self, plane_features):\n        batch, planes, channels, height, width = plane_features.shape\n        if (planes, height, width) != (self.planes, self.height, self.width):\n            raise RuntimeError(f'Unexpected bottleneck shape: {plane_features.shape}')\n        tokens = plane_features.permute(0, 1, 3, 4, 2).reshape(batch, planes * height * width, channels)\n        tokens = self.input_projection(tokens) + self.position\n        tokens = self.output_projection(self.transformer(tokens))\n        maps = tokens.reshape(batch, planes, height, width, channels).permute(0, 1, 4, 2, 3)\n        return self.fusion(maps.reshape(batch, planes * channels, height, width))\n\nclass FourierExpert(nn.Module):\n    def __init__(self, channels=512, planes=4, spectral_channels=128, height=8, width=8):\n        super().__init__()\n        self.planes = planes\n        self.reduce = nn.Conv2d(channels, spectral_channels, 1, bias=False)\n        self.weight_real = nn.Parameter(torch.ones(1, spectral_channels, height, width // 2 + 1))\n        self.weight_imag = nn.Parameter(torch.zeros(1, spectral_channels, height, width // 2 + 1))\n        self.expand = nn.Sequential(\n            nn.Conv2d(spectral_channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU(),\n        )\n        self.fusion = PlaneFusion(channels)\n\n    def forward(self, plane_features):\n        batch, planes, channels, height, width = plane_features.shape\n        values = self.reduce(plane_features.reshape(batch * planes, channels, height, width))\n        spectrum = torch.fft.rfft2(values.float(), norm='ortho')\n        spectral_weight = torch.complex(self.weight_real, self.weight_imag)\n        values = torch.fft.irfft2(spectrum * spectral_weight, s=(height, width), norm='ortho')\n        values = self.expand(values.to(plane_features.dtype))\n        values = values.reshape(batch, planes, channels, height, width)\n        return self.fusion(values.reshape(batch, planes * channels, height, width))\n\nclass HarmonicMoEUNet(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.planes = 4\n        channels = (32, 64, 128, 256, 512)\n        self.encoders = nn.ModuleList([\n            DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128),\n            DoubleConv(128, 256), DoubleConv(256, 512),\n        ])\n        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])\n        self.fusions = nn.ModuleList([PlaneFusion(c) for c in channels])\n        self.pool = nn.MaxPool2d(2)\n        self.spatial_expert = SpatialExpert(512)\n        self.transformer_expert = CrossPlaneTransformerExpert(512)\n        self.fourier_expert = FourierExpert(512)\n        self.gate = nn.Sequential(nn.Linear(16 + 3 * 512, 256), nn.SiLU(), nn.Linear(256, 3))\n        self.d3, self.d2 = UpBlock(512, 256, 256), UpBlock(256, 128, 128)\n        self.d1, self.d0 = UpBlock(128, 64, 64), UpBlock(64, 32, 32)\n        self.head = nn.Conv2d(32, 1, 1)\n        z = torch.tensor(ZS, dtype=torch.float32) * 1e3\n        self.register_buffer('depths', (z - z.mean()) / (z.max() - z.min()))\n\n    @staticmethod\n    def image_statistics(x):\n        image = (x + 1) * 0.5\n        mean = image.mean(dim=(-2, -1))\n        std = image.std(dim=(-2, -1))\n        dark = (image < 0.05).to(image.dtype).mean(dim=(-2, -1))\n        dx = (image[..., :, 1:] - image[..., :, :-1]).abs().mean(dim=(-2, -1))\n        dy = (image[..., 1:, :] - image[..., :-1, :]).abs().mean(dim=(-2, -1))\n        edge = 0.5 * (dx + dy)\n        return torch.stack((mean, std, dark, edge), dim=-1).flatten(1)\n\n    def forward(self, x):\n        batch, planes, height, width = x.shape\n        features = x.reshape(batch * planes, 1, height, width)\n        depths = self.depths.repeat(batch)\n        skips = []\n        plane_bottleneck = None\n        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):\n            features = encoder(features) + embedding(depths).to(features.dtype)\n            _, channels, h, w = features.shape\n            planes_view = features.reshape(batch, planes, channels, h, w)\n            if level < 4:\n                skips.append(fusion(planes_view.reshape(batch, planes * channels, h, w)))\n                features = self.pool(features)\n            else:\n                plane_bottleneck = planes_view\n\n        plain_fused = self.fusions[4](plane_bottleneck.reshape(batch, planes * 512, 8, 8))\n        expert_outputs = torch.stack((\n            self.spatial_expert(plain_fused),\n            self.transformer_expert(plane_bottleneck),\n            self.fourier_expert(plane_bottleneck),\n        ), dim=1)\n        pooled = expert_outputs.mean(dim=(-2, -1)).flatten(1)\n        gates = torch.softmax(self.gate(torch.cat((self.image_statistics(x), pooled), dim=1)), dim=1)\n        fused = (expert_outputs * gates[:, :, None, None, None]).sum(dim=1)\n        e0, e1, e2, e3 = skips\n        fused = self.d3(fused, e3); fused = self.d2(fused, e2)\n        fused = self.d1(fused, e1); fused = self.d0(fused, e0)\n        return self.head(fused), gates\n\nclass FresnelLayer(nn.Module):\n    def __init__(self, device):\n        super().__init__(); c = (torch.arange(IMAGE_SIZE, dtype=torch.float64) - IMAGE_SIZE / 2) * PIXEL_PITCH\n        u, v = torch.meshgrid(c, c, indexing='ij')\n        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]\n        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)\n    def forward(self, phase):\n        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float()); outputs = []\n        for kernel in self.kernels:\n            value = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()\n            outputs.append(value)\n        amplitude = torch.cat(outputs, dim=1)\n        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)\n\ndef pearson_cc(a, b, eps=1e-8):\n    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)\n    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)\n\ndef doe_metrics(target, reconstruction):\n    cc = pearson_cc(target, reconstruction)\n    sv = torch.stack([\n        ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False)\n        for m in range(4)\n    ], dim=1)\n    return cc, sv\n\nclass PhaseHarmonizer(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.e0, self.e1, self.e2 = DoubleConv(13, 32), DoubleConv(32, 64), DoubleConv(64, 128)\n        self.pool = nn.MaxPool2d(2)\n        self.bottleneck = DoubleConv(128, 128)\n        self.u2, self.u1, self.u0 = UpBlock(128, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)\n        self.head = nn.Conv2d(32, 2, 1)\n        nn.init.zeros_(self.head.weight)\n        nn.init.zeros_(self.head.bias)\n        with torch.no_grad(): self.head.bias[0] = 1.0\n\n    def forward(self, target, initial_reconstruction, initial_phase):\n        error = target - initial_reconstruction\n        x = torch.cat((target, initial_reconstruction, error, initial_phase / math.pi), dim=1)\n        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1))\n        x = self.bottleneck(self.pool(e2)); x = self.u2(x, e2); x = self.u1(x, e1); x = self.u0(x, e0)\n        raw = self.head(x); norm = torch.sqrt(raw[:, :1].square() + raw[:, 1:].square() + 1e-8)\n        return raw[:, :1] / norm, raw[:, 1:] / norm\n\nclass EndToEndHarmonized(nn.Module):\n    def __init__(self, device):\n        super().__init__(); self.backbone = HarmonicMoEUNet(); self.harmonizer = PhaseHarmonizer()\n        self.physics = FresnelLayer(device)\n\n    def synthesize(self, target):\n        target = target.float()\n        with torch.autocast(device_type='cuda'):\n            logits, gates = self.backbone(2 * target - 1)\n            initial_phase = math.pi * torch.tanh(logits)\n        with torch.autocast(device_type='cuda', enabled=False):\n            initial_reconstruction = self.physics(initial_phase.float())\n        with torch.autocast(device_type='cuda'):\n            correction_real, correction_imag = self.harmonizer(target, initial_reconstruction, initial_phase)\n        base = torch.polar(torch.ones_like(initial_phase, dtype=torch.float32), initial_phase.float())\n        correction = torch.complex(correction_real.float(), correction_imag.float())\n        final_field = base * correction\n        final_phase = torch.atan2(final_field.imag, final_field.real)\n        with torch.autocast(device_type='cuda', enabled=False):\n            final_reconstruction = self.physics(final_phase)\n        return initial_phase, initial_reconstruction, final_phase, final_reconstruction, gates, correction_real\n\n    def forward(self, target):\n        initial_phase, initial_reconstruction, final_phase, final_reconstruction, gates, correction_real = self.synthesize(target)\n        target = target.float(); initial_cc, initial_ssim = doe_metrics(target, initial_reconstruction)\n        final_cc, final_ssim = doe_metrics(target, final_reconstruction)\n        initial_loss = (1-initial_cc).mean() + SSIM_WEIGHT * (1-initial_ssim).mean()\n        final_loss = (1-final_cc).mean() + SSIM_WEIGHT * (1-final_ssim).mean()\n        correction_penalty = (1-correction_real).mean()\n        mean_gate = gates.float().mean(0); balance = (mean_gate-1/3).square().sum()\n        total = final_loss + 0.2*initial_loss + 0.01*correction_penalty + GATE_BALANCE_WEIGHT*balance\n        return total, final_cc, final_ssim, gates, correction_penalty, initial_cc, initial_ssim\n\ndef setup():\n    dist.init_process_group('nccl'); local_rank=int(os.environ['LOCAL_RANK']); torch.cuda.set_device(local_rank)\n    return local_rank,dist.get_rank(),dist.get_world_size(),torch.device(f'cuda:{local_rank}')\ndef unwrap(model): return model.module\ndef reduce_tensor(value): dist.all_reduce(value,op=dist.ReduceOp.SUM); return value\n\n@torch.no_grad()\ndef validate(model,loader,device):\n    model.eval(); stats=torch.zeros(27,device=device)\n    for target,sources in loader:\n        target=target.to(device,non_blocking=True).float(); sources=sources.to(device,non_blocking=True)\n        _,cc,sv,gates,corr,icc,isv=model(target); b=target.shape[0]\n        stats[0:4]+=cc.sum(0); stats[4:8]+=sv.sum(0); stats[8:11]+=gates.sum(0)\n        stats[11:15]+=icc.sum(0); stats[15:19]+=isv.sum(0)\n        for sid in (0,1):\n            mask=sources==sid; stats[19+sid]+=cc[mask].sum(); stats[21+sid]+=sv[mask].sum(); stats[23+sid]+=mask.sum()\n        stats[25]+=b; stats[26]+=corr*b\n    stats=reduce_tensor(stats); n=stats[25]; ccv=(stats[0:4]/n).cpu().tolist(); ssv=(stats[4:8]/n).cpu().tolist()\n    iccv=(stats[11:15]/n).cpu().tolist(); issv=(stats[15:19]/n).cpu().tolist(); gates=(stats[8:11]/n).cpu().tolist()\n    row={'val_loss':float(1-np.mean(ccv)+SSIM_WEIGHT*(1-np.mean(ssv))),'val_cc':float(np.mean(ccv)),\n         'val_ssim':float(np.mean(ssv)),'val_cc_min':float(np.min(ccv)),'val_ssim_min':float(np.min(ssv)),\n         'initial_cc':float(np.mean(iccv)),'initial_ssim':float(np.mean(issv)),'correction_penalty':float(stats[26]/n),\n         'gate_spatial':gates[0],'gate_transformer':gates[1],'gate_fourier':gates[2]}\n    row.update({f'cc_z{i+1}':v for i,v in enumerate(ccv)}); row.update({f'ssim_z{i+1}':v for i,v in enumerate(ssv)})\n    for sid,label in ((0,'synthetic'),(1,'bsds')):\n        row[f'cc_{label}']=float(stats[19+sid]/stats[23+sid]); row[f'ssim_{label}']=float(stats[21+sid]/stats[23+sid])\n    return row\n\ndef save(path,state):\n    tmp=path.with_suffix('.tmp'); torch.save(state,tmp); tmp.replace(path)\ndef main():\n    local_rank,rank,world,device=setup(); random.seed(SEED+rank); np.random.seed(SEED+rank); torch.manual_seed(SEED+rank)\n    train=CachedQuadDataset('train'); val=CachedQuadDataset('val')\n    ts=DistributedSampler(train,num_replicas=world,rank=rank,shuffle=True,seed=SEED)\n    vs=DistributedSampler(val,num_replicas=world,rank=rank,shuffle=False)\n    tl=DataLoader(train,batch_size=PER_GPU_BATCH,sampler=ts,num_workers=2,pin_memory=True,persistent_workers=True)\n    vl=DataLoader(val,batch_size=PER_GPU_BATCH,sampler=vs,num_workers=2,pin_memory=True,persistent_workers=True)\n    core=EndToEndHarmonized(device).to(device)\n    if BASE_CHECKPOINT is not None and RESUME_CHECKPOINT is None:\n        base=torch.load(BASE_CHECKPOINT,map_location=device,weights_only=False); core.backbone.load_state_dict(base['model'],strict=False)\n    model=DDP(core,device_ids=[local_rank],output_device=local_rank,find_unused_parameters=False,gradient_as_bucket_view=True)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS,eta_min=MIN_LR)\n    scaler=torch.amp.GradScaler('cuda'); history=[]; start=0; stale=0\n    best={'joint_loss':float('inf'),'cc':-float('inf'),'ssim':-float('inf')}\n    if RESUME_CHECKPOINT is not None:\n        ck=torch.load(RESUME_CHECKPOINT,map_location=device,weights_only=False); unwrap(model).load_state_dict(ck['model'])\n        optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])\n        history=ck['history']; best=ck['best']; stale=ck['stale']; start=ck['epoch']+1\n    for epoch in range(start,MAX_EPOCHS):\n        ts.set_epoch(epoch); model.train(); total=0.; seen=0; started=time.perf_counter()\n        for bi,(target,_) in enumerate(tl,1):\n            target=target.to(device,non_blocking=True).float(); optimizer.zero_grad(set_to_none=True)\n            loss,*_=model(target); scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()\n            total+=loss.item()*target.shape[0]; seen+=target.shape[0]\n            if rank==0 and (bi==1 or bi%50==0 or bi==len(tl)):\n                eta=(time.perf_counter()-started)/bi*(len(tl)-bi); print(f'epoch {epoch+1} batch {bi}/{len(tl)} loss={loss.item():.4f} ETA={eta/60:.1f}m',flush=True)\n        row=validate(model,vl,device); scheduler.step(); row.update({'epoch':epoch+1,'train_loss':total/seen,'lr':optimizer.param_groups[0]['lr'],'epoch_seconds':time.perf_counter()-started})\n        ij=row['val_loss']<best['joint_loss']; ic=row['val_cc']>best['cc']; iss=row['val_ssim']>best['ssim']\n        if ij: best['joint_loss']=row['val_loss']; stale=0\n        else: stale+=1\n        if ic: best['cc']=row['val_cc']\n        if iss: best['ssim']=row['val_ssim']\n        if rank==0:\n            history.append(row); state={'epoch':epoch,'model':unwrap(model).state_dict(),'optimizer':optimizer.state_dict(),\n                'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'history':history,'best':best,'stale':stale,'metrics_at_save':row}\n            save(OUTPUT_DIR/'last.pt',state)\n            if ij: save(OUTPUT_DIR/'best_joint.pt',state)\n            if ic: save(OUTPUT_DIR/'best_cc.pt',state)\n            if iss: save(OUTPUT_DIR/'best_ssim.pt',state)\n            with (OUTPUT_DIR/'metrics.csv').open('w',newline='') as f:\n                w=csv.DictWriter(f,fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)\n            print(row,flush=True)\n        stop=torch.tensor(int(epoch+1>=MIN_EPOCHS and stale>=PATIENCE),device=device); dist.broadcast(stop,src=0)\n        if stop.item(): break\n    dist.barrier()\n    if rank==0:\n        ck=torch.load(OUTPUT_DIR/'best_joint.pt',map_location=device,weights_only=False); unwrap(model).load_state_dict(ck['model']); unwrap(model).eval()\n        target,sources=val[0]; target=target.float().unsqueeze(0).to(device)\n        with torch.no_grad(): _,_,phase,recon,gates,_=unwrap(model).synthesize(target)\n        out=OUTPUT_DIR/'validation_export'; out.mkdir(exist_ok=True); doe=((phase[0,0]+math.pi)/(2*math.pi)*255).round().clamp(0,255).byte().cpu().numpy()\n        Image.fromarray(doe,mode='L').save(out/'doe.png'); (OUTPUT_DIR/'run_complete.txt').write_text('complete')\n    dist.barrier(); dist.destroy_process_group()\nif __name__=='__main__': main()\n"
DDP_SCRIPT = (DDP_SCRIPT_TEMPLATE
    .replace('__PER_GPU_BATCH__', str(PER_GPU_BATCH))
    .replace('__BASE_CHECKPOINT__', repr(DDP_BASE_CHECKPOINT))
    .replace('__RESUME_CHECKPOINT__', repr(DDP_RESUME_CHECKPOINT)))
DDP_SCRIPT_PATH = Path('/kaggle/working/train_harmonic_moe_ddp.py')
DDP_SCRIPT_PATH.write_text(DDP_SCRIPT, encoding='utf-8')
print('Wrote:', DDP_SCRIPT_PATH, 'characters:', len(DDP_SCRIPT))

In [ ]:
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', str(DDP_SCRIPT_PATH)]
print('Launching:', ' '.join(command), flush=True)
subprocess.run(command, check=True)
print((OUTPUT_DIR / 'metrics.csv').read_text().splitlines()[-1])